In [1]:

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

os.makedirs("figures", exist_ok=True)

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.color":       "#e5e5e5",
    "grid.linewidth":   0.6,
    "figure.dpi":       150,
    "savefig.dpi":      200,
    "savefig.bbox":     "tight",
    "savefig.facecolor":"white",
})

BLUE   = "#2563A8"
TEAL   = "#1D9E75"
AMBER  = "#D4790A"
CORAL  = "#C0392B"
GRAY   = "#6B7280"
PURPLE = "#7C4DBC"

epochs = list(range(1, 31))
bpr_loss = [
    4.8940, 3.6986, 2.6529, 1.7276, 0.9374,
    0.6592, 0.5826, 0.5157, 0.4649, 0.4169,
    0.3681, 0.3247, 0.2844, 0.2507, 0.2190,
    0.2000, 0.1698, 0.1505, 0.1367, 0.1223,
    0.1150, 0.1040, 0.0970, 0.0890, 0.0880,
    0.0837, 0.0826, 0.0812, 0.0790, 0.0792,
]
# Cosine LR schedule values from notebook
lr_vals = [
    9.97e-4, 9.89e-4, 9.76e-4, 9.57e-4, 9.34e-4,
    9.05e-4, 8.73e-4, 8.36e-4, 7.96e-4, 7.53e-4,
    7.06e-4, 6.58e-4, 6.08e-4, 5.57e-4, 5.05e-4,
    4.53e-4, 4.02e-4, 3.52e-4, 3.04e-4, 2.58e-4,
    2.14e-4, 1.74e-4, 1.37e-4, 1.05e-4, 7.63e-5,
    5.28e-5, 3.42e-5, 2.08e-5, 1.27e-5, 1.00e-5,
]

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax2 = ax1.twinx()

ax1.plot(epochs, bpr_loss, color=BLUE, linewidth=2.2, marker="o",
         markersize=4, markevery=5, label="BPR Loss", zorder=3)
ax2.plot(epochs, lr_vals,  color=AMBER, linewidth=1.4, linestyle="--",
         alpha=0.7, label="Learning rate", zorder=2)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("BPR Loss", color=BLUE)
ax2.set_ylabel("Learning rate", color=AMBER)
ax1.tick_params(axis="y", labelcolor=BLUE)
ax2.tick_params(axis="y", labelcolor=AMBER)
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_color(AMBER)
ax2.spines["top"].set_visible(False)

ax1.annotate(f"0.079", xy=(30, 0.079), xytext=(27, 0.5),
             arrowprops=dict(arrowstyle="->", color=GRAY, lw=1),
             color=GRAY, fontsize=9)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

ax1.set_title("BPR training loss — TiSASRec full model (Text + Image)\n"
              "37,571 users · Adam + cosine annealing · 30 epochs",
              fontsize=11, pad=10)
ax1.set_xlim(1, 30)
ax1.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("figures/01_training_loss.png")
plt.close()
print("✔  Saved figures/01_training_loss.png")
tiers  = ["Cold\n(1,045)", "Few-shot\n(1,930)", "Warm\n(5,176)", "Overall\n(8,151)"]
r_at5  = [0.0010, 0.0378, 0.3331, 0.2206]
r_at10 = [0.0153, 0.0922, 0.4635, 0.3181]
r_at20 = [0.0842, 0.2021, 0.5875, 0.4317]
n_at5  = [0.0005, 0.0208, 0.2328, 0.1528]
n_at10 = [0.0049, 0.0380, 0.2747, 0.1841]
n_at20 = [0.0218, 0.0654, 0.3063, 0.2128]

x      = np.arange(len(tiers))
width  = 0.13

fig, (ax_r, ax_n) = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, vals5, vals10, vals20, ylabel, title in [
    (ax_r, r_at5, r_at10, r_at20, "Recall", "Recall@K by tier"),
    (ax_n, n_at5, n_at10, n_at20, "NDCG",  "NDCG@K by tier"),
]:
    b1 = ax.bar(x - width, vals5,  width, label="@5",  color=BLUE,   alpha=0.75)
    b2 = ax.bar(x,          vals10, width, label="@10", color=TEAL,   alpha=0.85)
    b3 = ax.bar(x + width,  vals20, width, label="@20", color=PURPLE, alpha=0.75)
    ax.set_xticks(x)
    ax.set_xticklabels(tiers, fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)
    # value labels on top of warm / overall bars only (others too small)
    for bars in [b1, b2, b3]:
        for i, bar in enumerate(bars):
            if bar.get_height() > 0.05:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.008,
                        f"{bar.get_height():.2f}",
                        ha="center", va="bottom", fontsize=7.5, color=GRAY)

fig.suptitle("Full model evaluation by user tier — TiSASRec (Text + Image)\n"
             "100-way sampled ranking · Amazon Beauty test set",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig("figures/02_tier_performance.png")
plt.close()
print("✔  Saved figures/02_tier_performance.png")

variants = ["ID only", "Text only", "Image only", "Text + Image"]

# From ablation notebook summary (20% stratified subsample)
abl_overall_r10 = [0.0135, 0.0074, 0.0080, 0.0098]
abl_overall_n10 = [0.0064, 0.0032, 0.0042, 0.0042]
abl_cold_r10    = [0.0148, 0.0089, 0.0059, 0.0118]
abl_cold_n10    = [0.0026, 0.0027, 0.0026, 0.0051]
abl_few_r10     = [0.0182, 0.0120, 0.0182, 0.0000]
abl_few_n10     = [0.0120, 0.0096, 0.0120, 0.0000]

x      = np.arange(len(variants))
width  = 0.35

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
configs = [
    (axes[0], abl_overall_r10, abl_overall_n10, "Overall"),
    (axes[1], abl_cold_r10,    abl_cold_n10,    "Cold tier"),
    (axes[2], abl_few_r10,     abl_few_n10,     "Few-shot tier"),
]

for ax, r10, n10, title in configs:
    bars_r = ax.bar(x - width/2, r10, width, label="Recall@10", color=BLUE,  alpha=0.85)
    bars_n = ax.bar(x + width/2, n10, width, label="NDCG@10",   color=TEAL,  alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(["ID\nonly", "Text\nonly", "Image\nonly", "Text+\nImage"],
                       fontsize=9)
    ax.set_title(title, fontsize=11, pad=6)
    ax.set_ylabel("Score")
    ax.legend(fontsize=8)
    # value labels
    for bars in [bars_r, bars_n]:
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.0002,
                        f"{h:.4f}", ha="center", va="bottom",
                        fontsize=7, color=GRAY, rotation=45)

fig.suptitle("Ablation study — impact of modality on Recall@10 & NDCG@10\n"
             "20% stratified subsample · Amazon Beauty · TiSASRec backbone",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("figures/03_ablation_study.png")
plt.close()
print("✔  Saved figures/03_ablation_study.png")
recall_data = np.array([
    [0.0044, 0.0148,  0.0296,  0.0109, 0.0182, 0.0328, 0.0055, 0.0135, 0.0252],  # ID only
    [0.0015, 0.0089,  0.0163,  0.0073, 0.0120, 0.0219, 0.0025, 0.0074, 0.0141],  # Text only
    [0.0044, 0.0059,  0.0148,  0.0109, 0.0182, 0.0328, 0.0055, 0.0080, 0.0178],  # Image only
    [0.0059, 0.0118,  0.0222,  0.0000, 0.0000, 0.0109, 0.0049, 0.0098, 0.0202],  # Text+Image
])

col_labels = [
    "Cold\n@5", "Cold\n@10", "Cold\n@20",
    "Few\n@5",  "Few\n@10",  "Few\n@20",
    "Ovr\n@5",  "Ovr\n@10",  "Ovr\n@20",
]
row_labels = ["ID only", "Text only", "Image only", "Text + Image"]

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(recall_data, aspect="auto", cmap="Blues", vmin=0, vmax=0.035)

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=9)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=10)

# Annotate cells
for i in range(recall_data.shape[0]):
    for j in range(recall_data.shape[1]):
        val = recall_data[i, j]
        color = "white" if val > 0.018 else "black"
        ax.text(j, i, f"{val:.4f}", ha="center", va="center",
                fontsize=8.5, color=color, fontweight="normal")

# Column group dividers
for x_pos in [2.5, 5.5]:
    ax.axvline(x=x_pos, color="white", linewidth=2)

plt.colorbar(im, ax=ax, label="Recall@K", shrink=0.8)
ax.set_title("Recall heatmap — ablation study across tiers and K\n"
             "Amazon Beauty · 20% subsample",
             fontsize=11, pad=10)

# Group labels above
for label, xpos in [("Cold tier", 1), ("Few-shot tier", 4), ("Overall", 7)]:
    ax.text(xpos, -0.9, label, ha="center", va="center",
            fontsize=9, color=GRAY, transform=ax.transData)

plt.tight_layout()
plt.savefig("figures/04_recall_heatmap.png")
plt.close()
print("✔  Saved figures/04_recall_heatmap.png")

fig = plt.figure(figsize=(12, 4.5))
gs  = GridSpec(1, 2, figure=fig, wspace=0.35)

# -- Left: stacked horizontal bar for train/val/test split --
ax_l = fig.add_subplot(gs[0])
split_counts = {"Train": 677545, "Val": 8151, "Test": 8151}
total        = sum(split_counts.values())
colors_split = [BLUE, TEAL, AMBER]
left         = 0
for (label, count), color in zip(split_counts.items(), colors_split):
    pct = count / total * 100
    bar = ax_l.barh(0, pct, left=left, color=color, height=0.5)
    if pct > 2:
        ax_l.text(left + pct / 2, 0, f"{label}\n{count:,}\n({pct:.1f}%)",
                  ha="center", va="center", fontsize=9, color="white", fontweight="500")
    left += pct

ax_l.set_xlim(0, 100)
ax_l.set_yticks([])
ax_l.set_xlabel("% of interactions")
ax_l.set_title("Interaction split (leave-one-out)\n701,528 total", fontsize=10, pad=8)
ax_l.spines["left"].set_visible(False)

# -- Right: test tier pie --
ax_r = fig.add_subplot(gs[1])
tier_sizes  = [1045, 1930, 5176]
tier_labels = ["Cold\n(1,045)", "Few-shot\n(1,930)", "Warm\n(5,176)"]
tier_colors = [BLUE, AMBER, TEAL]
wedges, texts, autotexts = ax_r.pie(
    tier_sizes, labels=tier_labels, colors=tier_colors,
    autopct="%1.1f%%", startangle=90,
    wedgeprops={"linewidth": 0.8, "edgecolor": "white"},
    textprops={"fontsize": 9},
)
for at in autotexts:
    at.set_fontsize(8.5)
    at.set_color("white")
ax_r.set_title("Test user tier distribution\n8,151 test users", fontsize=10, pad=8)

fig.suptitle("Amazon Beauty 2023 — Dataset Statistics\n"
             "631K users · 112K items · 701K interactions",
             fontsize=11, y=1.02)
plt.savefig("figures/05_dataset_stats.png")
plt.close()
print("✔  Saved figures/05_dataset_stats.png")


print("\nAll done! Check the ./figures/ folder for:")
print("  01_training_loss.png      — BPR loss + LR curve")
print("  02_tier_performance.png   — Recall & NDCG by tier (full model)")
print("  03_ablation_study.png     — Ablation: R@10 & N@10 by modality")
print("  04_recall_heatmap.png     — Recall heatmap across tiers × K")
print("  05_dataset_stats.png      — Dataset split + test tier pie")

✔  Saved figures/01_training_loss.png
✔  Saved figures/02_tier_performance.png
✔  Saved figures/03_ablation_study.png
✔  Saved figures/04_recall_heatmap.png
✔  Saved figures/05_dataset_stats.png

All done! Check the ./figures/ folder for:
  01_training_loss.png      — BPR loss + LR curve
  02_tier_performance.png   — Recall & NDCG by tier (full model)
  03_ablation_study.png     — Ablation: R@10 & N@10 by modality
  04_recall_heatmap.png     — Recall heatmap across tiers × K
  05_dataset_stats.png      — Dataset split + test tier pie
